## PINNs: First order PDE on 2d

__Problem__

Let $a \neq 0$ and consider an equation of 
$$a u_x + b u_y = 0; \ u(0,y) = g(y).$$

Its exact solution is:
$$u(x, y) = g(y - \frac{b}{a}x).$$

__PDE setup__

- Domain: $\Omega = [0,1]\times[0,1]$
- Inflow boundary: $\partial^0\Omega = \{0\}\times[0,1]$
- Equation:
    $$
    a\,u_x(x,y) + b\,u_y(x,y) = 0, \quad (x,y)\in\Omega
    $$
- Boundary condition:
    $$
    u(0,y)=g(y), \quad y\in[0,1]
    $$

Using $p=\nabla u=(u_x,u_y)$ and $\bar g(x,y) = g(y)$, we write
$$
F(p)=(a,b)^\top\!\cdot p,\qquad B(u)=u - \bar g,
$$
the residual functional is
$$
G(u)=\|F(\nabla u)\|_{L^2(\Omega)}^2+\|B(u)\|_{L^2(\partial^0\Omega)}^2.
$$

A function $u$ solves the problem exactly iff $G(u)=0$.


To implement this idea, define a neural-network hypothesis class
$$
\mathcal{N}=\{h_\theta:\theta\in\mathbb{R}^m\},
$$
and determine $\theta^*$ by minimizing
$$
\theta^*=\arg\min_\theta\, G\!\left(h_\theta\right).
$$

Then $h_{\theta^*}$ serves as the PINN approximation of $u$.

In [1]:
# code starts here
import torch

import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# Reproducibility
torch.manual_seed(42)

In [28]:
# Create a class for PDE
class PDE_1order:
    def __init__(self, a=4.0, b=-3.0, g=lambda y: torch.sin(y)):
        self.a = a
        self.b = b
        self.g = g
        self.g_name = "sin(y)"

    # Print the PDE
    def __str__(self):
        return f"({self.a})u_1 + ({self.b})u_2 = 0; u(0,y) = {self.g_name}"

    # PINNs model
    def pinns_model(self):
        return nn.Sequential(
            nn.Linear(2, 20),  # Input layer (x, y)
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 1)   # Output layer (u)
        )

    # Loss function for PDE residual
    def loss_function(self, model, x):
        x = x.clone().detach().requires_grad_(True)
        u = model(x)
        grads = torch.autograd.grad(
            u, x, grad_outputs=torch.ones_like(u), create_graph=True
        )[0]
        u_1 = grads[:, 0:1]
        u_2 = grads[:, 1:2]
        residual = self.a * u_1 + self.b * u_2
        return torch.mean(residual**2)

    # Loss function for boundary condition
    def boundary_loss(self, model, y):
        x1 = torch.zeros_like(y)  # x1 = 0 for boundary condition
        inputs = torch.cat((x1, y), dim=1)
        u_pred = model(inputs)
        g_y = self.g(y)
        return torch.mean((u_pred - g_y) ** 2)

    # Training loop
    def train(self, model, x_collocation, y_boundary, epochs=1000, lr=0.001):
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        for epoch in range(epochs):
            optimizer.zero_grad()
            loss_pde = self.loss_function(model, x_collocation)
            loss_bc = self.boundary_loss(model, y_boundary)
            loss = loss_pde + loss_bc
            loss.backward()
            optimizer.step()
            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")

    # PDE solver using PINNs
    def solve(
        self,
        x1_range=(0, 1),
        x2_range=(0, 1),
        num_collocation=1000,
        num_boundary=100,
        epochs=1000,
        lr=0.001
    ):
        x1_min, x1_max = x1_range
        x2_min, x2_max = x2_range

        # Generate collocation points in [x1_min, x1_max] x [x2_min, x2_max]
        x_collocation = torch.rand(num_collocation, 2)
        x_collocation[:, 0] = x1_min + (x1_max - x1_min) * x_collocation[:, 0]
        x_collocation[:, 1] = x2_min + (x2_max - x2_min) * x_collocation[:, 1]

        # Generate boundary points y in [x2_min, x2_max] for x=0
        y_boundary = x2_min + (x2_max - x2_min) * torch.rand(num_boundary, 1)

        model = self.pinns_model()
        self.train(model, x_collocation, y_boundary, epochs, lr)
        return model


# create an instance of the PDE
pde = PDE_1order(a=4.0, b=-3.0, g=lambda y: torch.sin(y))
print(pde)

# solve the PDE using PINNs
model = pde.solve(epochs=5000, lr=0.001)


(4.0)u_1 + (-3.0)u_2 = 0; u(0,y) = sin(y)
Epoch 100/5000, Loss: 0.105882
Epoch 200/5000, Loss: 0.062106
Epoch 300/5000, Loss: 0.044557
Epoch 400/5000, Loss: 0.028195
Epoch 500/5000, Loss: 0.011987
Epoch 600/5000, Loss: 0.003076
Epoch 700/5000, Loss: 0.000964
Epoch 800/5000, Loss: 0.000512
Epoch 900/5000, Loss: 0.000334
Epoch 1000/5000, Loss: 0.000243
Epoch 1100/5000, Loss: 0.000191
Epoch 1200/5000, Loss: 0.000159
Epoch 1300/5000, Loss: 0.000137
Epoch 1400/5000, Loss: 0.000120
Epoch 1500/5000, Loss: 0.000107
Epoch 1600/5000, Loss: 0.000097
Epoch 1700/5000, Loss: 0.000088
Epoch 1800/5000, Loss: 0.000080
Epoch 1900/5000, Loss: 0.000073
Epoch 2000/5000, Loss: 0.000067
Epoch 2100/5000, Loss: 0.000062
Epoch 2200/5000, Loss: 0.000057
Epoch 2300/5000, Loss: 0.000052
Epoch 2400/5000, Loss: 0.000048
Epoch 2500/5000, Loss: 0.000044
Epoch 2600/5000, Loss: 0.000040
Epoch 2700/5000, Loss: 0.000037
Epoch 2800/5000, Loss: 0.000034
Epoch 2900/5000, Loss: 0.000031
Epoch 3000/5000, Loss: 0.000028
Epoch 3